In [4]:
"""Mapping recharge to precipitation using dynamic time warping (DTW)"""

#%%
import numpy as np
import pandas as pd
from dtw import dtw

import matplotlib
matplotlib.use('QtAgg') 
import matplotlib.pyplot as plt

#%%

Importing the dtw module. When using in academic works please cite:
  T. Giorgino. Computing and Visualizing Dynamic Time Warping Alignments in R: The dtw Package.
  J. Stat. Soft., doi:10.18637/jss.v031.i07.



In [ ]:
# Load recharge and precipitation data for a single site
site_id = "402750078452201"

df_recharge = pd.read_csv(f"C:/Users/romin/OneDrive/Groundwater/RpSy Data/{site_id}.csv")
df_precip = pd.read_csv(f"./daymet/{site_id}.csv")

#convert year and yday to datetime
df_precip["Date"] = pd.to_datetime(df_precip['year'] * 1000 + df_precip['yday'], format='%Y%j')
df_recharge["Date"] = pd.to_datetime(df_recharge["Date"])

# set index and merge the two dataframes on the Date column
df_recharge.set_index("Date", inplace=True)
df_precip.set_index("Date", inplace=True)

df = pd.merge(df_recharge, df_precip, on="Date", how="inner")
df.drop(columns=["year", "yday"], inplace=True)


In [19]:
#plot recharge on the left y-axis and precipitation on the right y-axis
# for best results use the QtAgg backend for matplotlib (see above)
fig, ax1 = plt.subplots(figsize=(10, 5))
color = 'tab:blue'
ax1.set_xlabel('Date')
ax1.set_ylabel('Recharge (m)', color="red")
ax1.plot(df.index, df["RpSy (m)"], color=color)
ax1.tick_params(axis='y', labelcolor=color)

ax2 = ax1.twinx()  # instantiate a second axes that shares the same x-axis
color = 'tab:red'
ax2.set_ylabel('Precipitation (mm)', color="blue")  # we already handled the x-label with ax1
ax2.plot(df.index, df["prcp (mm/day)"], color=color)
ax2.tick_params(axis='y', labelcolor=color)
ax2.set_ylim(100, 0)
plt.show()

In [ ]:
# testing with a smaller time window
df_plot = df.loc["2010-01-01":"2010-06-01", ["RpSy (m)", "prcp (mm/day)"]]

#plot recharge on the left y-axis and precipitation on the right y-axis
fig, ax1 = plt.subplots(figsize=(10, 5))
color = 'tab:blue'
ax1.set_xlabel('Date')
ax1.set_ylabel('Recharge (m)', color=color)
ax1.plot(df_plot.index, df_plot["RpSy (m)"], color=color)
ax1.tick_params(axis='y', labelcolor=color)

ax2 = ax1.twinx()  # instantiate a second axes that shares the same x-axis
color = 'tab:red'
ax2.set_ylabel('Precipitation (mm)', color=color)  # we already handled the x-label with ax1
ax2.plot(df_plot.index, df_plot["prcp (mm/day)"], color=color)
ax2.tick_params(axis='y', labelcolor=color)
ax2.set_ylim(100, 0)
plt.show()


In [ ]:
#%% Vanilla DTW

# extract timeseries values and normalize them by their standard deviation
# if timeseries are not normalized, DTW will not find the best alignment
recharge = df_plot["RpSy (m)"].values 
recharge_norm = (recharge)/np.std(recharge)  # center recharge around zero
precip = df_plot["prcp (mm/day)"].values
precip_norm = (precip)/np.std(precip)  # center precipitation around zero

large_precip = np.where(precip_norm > 1)[0]  # indices of large precipitation events
large_recharge = np.where(recharge_norm > 1)[0]  # indices of large recharge events

#vanilla DTW
alignment = dtw(
    recharge_norm, precip_norm,
    keep_internals=True,
    open_begin=False,   # let recharge "start late" relative to precip record
    open_end=False,
)

# positions along the warping curve where the reference (precip) index
# is one of your large-precip events
match_pos = np.where(np.isin(alignment.index2, large_precip))[0]

# alignmnent plot, picking indices only of the large precipitation events to match to recharge
alignment.plot(type="twoway", offset=5, match_indices=match_pos)
plt.show()

In [14]:
#%% Advanced DTW with causal window

# query = recharge series, reference = precipitation series
# iw = index into query (recharge), jw = index into reference (precip)
# convention confirmed from the R/dtw source: iw=row=query, jw=col=reference

# use whole timeseries
df_plot = df.loc[:, ["RpSy (m)", "prcp (mm/day)"]]

recharge = df_plot["RpSy (m)"].values 
recharge_norm = (recharge)/np.std(recharge)  # center recharge around zero
precip = df_plot["prcp (mm/day)"].values
precip_norm = (precip)/np.std(precip)  # center precipitation around zero

large_precip = np.where(precip_norm > 0.5)[0]  # indices of large precipitation events
large_recharge = np.where(recharge_norm > 0.5)[0]  # indices of large recharge events
all_pos_precip = np.where(precip_norm > 0)[0]  # indices of all precipitation events

# this window only alows lags in some arbitrary window
# The intent is for min lag to be zero (recharge after precip)
# This ensures physically meaningful results.
def causal_window(iw, jw, query_size, reference_size, min_lag=0, max_lag=None, **kwargs):
    lag = iw - jw          # recharge index minus precip index
    ok = lag >= min_lag    # recharge can't precede its precip
    if max_lag is not None:
        ok = ok & (lag <= max_lag)   # optional cap on plausible response time
    return ok

alignment = dtw(
    recharge_norm, precip_norm,
    step_pattern="symmetric2",
    window_type=causal_window,
    window_args={"min_lag": 0, "max_lag": 5},
    keep_internals=True,
    open_begin=False,   # ideally would want to let recharge start late, but need asymmetric step pattern, not yet attempted here.
    open_end=False,
)
#%%

# positions along the warping curve where the reference (precip) index
# is one of your large-precip events
match_pos = np.where(np.isin(alignment.index2, all_pos_precip))[0]

alignment.plot(type="twoway", offset=5, match_indices=match_pos)
plt.show()

In [15]:
def get_events(values, threshold):
    """Return start/end index pairs for runs of consecutive values > threshold."""
    above = (values > threshold).astype(int)
    padded = np.concatenate(([0], above, [0]))
    diff = np.diff(padded)
    starts = np.where(diff == 1)[0]
    ends = np.where(diff == -1)[0] - 1
    return starts, ends

def precip_recharge_event_table(df, precip_col, recharge_col, alignment,
                                 precip_threshold=0.0):
    """Return a table of precipitation and recharge events based on DTW alignment."""

    dates = df.index
    precip_vals = df[precip_col].values
    recharge_vals = df[recharge_col].values

    starts, ends = get_events(precip_vals, precip_threshold)

    rows = []
    for s, e in zip(starts, ends):
        precip_idx = np.arange(s, e + 1)

        # alignment.index2 = precip (reference), alignment.index1 = recharge (query)
        mask = np.isin(alignment.index2, precip_idx)
        recharge_idx = np.unique(alignment.index1[mask])

        # you can still get single day overlaps, where a recharge event ends and starts on the same day
        # to avoid double counting, we can drop recharge indices that overlap with previous recharge events
        if rows:
            if prev_recharge_idx is not None:
                test = recharge_idx > prev_recharge_idx.max()
                if (test==False).any():
                    print(f"Warning: dropping {np.sum(~test)} recharge indices that overlap with previous recharge event")
                    recharge_idx = recharge_idx[test]

        row = {
            "precip_start_date": dates[s],
            "precip_end_date": dates[e],
            "precip_total": precip_vals[precip_idx].sum(),
            "precip_peak": precip_vals[precip_idx].max(),
        }

        if recharge_idx.size == 0:
            row.update({
                "recharge_start_date": pd.NaT,
                "recharge_end_date": pd.NaT,
                "recharge_total": np.nan,
                "recharge_peak": np.nan,
                "n_recharge_days": 0,
            })
        else:
            row.update({
                "recharge_start_date": dates[recharge_idx.min()],
                "recharge_end_date": dates[recharge_idx.max()],
                "recharge_total": recharge_vals[recharge_idx].sum(),
                "recharge_peak": recharge_vals[recharge_idx].max(),
                "n_recharge_days": recharge_idx.size,
            })

        rows.append(row)

        prev_recharge_idx = recharge_idx if recharge_idx.size > 0 else None

    return pd.DataFrame(rows)

event_table = precip_recharge_event_table(
    df_plot, "prcp (mm/day)", "RpSy (m)", alignment, precip_threshold=2.5
)



In [16]:
# check to make sure the above did it's job -- no overlapping recharge events
def check_overlapping_recharge_events(event_table):
    """Check for overlapping recharge events in the event table."""
    sorted_table = event_table.sort_values("recharge_start_date")
    overlaps = []
    for i in range(len(sorted_table) - 1):
        current_end = sorted_table.iloc[i]["recharge_end_date"]
        next_start = sorted_table.iloc[i + 1]["recharge_start_date"]
        if pd.notna(current_end) and pd.notna(next_start) and current_end >= next_start:
            overlap_days = (current_end - next_start).days + 1
            overlaps.append((i,current_end, next_start, overlap_days))
    return overlaps

overlapping_events = check_overlapping_recharge_events(event_table)
print(overlapping_events)

# %%

# scatter plot of total recharge vs total precipitation, colored by peak precipitation
plt.figure()
plt.scatter(event_table["precip_total"], event_table["recharge_total"], c=event_table["precip_peak"], alpha=0.5, s=30)
plt.xlabel("Total Precipitation (mm)")
plt.ylabel("Total Recharge (m)")
plt.colorbar(label="Peak Precipitation (mm/day)")
plt.show()

# %%

#plot recharge on the left y-axis and precipitation on the right y-axis
fig, ax1 = plt.subplots(figsize=(10, 5))
color = 'tab:blue'
ax1.set_xlabel('Date')
ax1.set_ylabel('Recharge (m)', color=color)
ax1.plot(df_plot.index, df_plot["RpSy (m)"], color=color)
ax1.tick_params(axis='y', labelcolor=color)
ax1.set_ylim(0.0, 0.7) # may need to adjust this depending on the site and time window

ax2 = ax1.twinx()  # instantiate a second axes that shares the same x-axis
color = 'tab:red'
ax2.set_ylabel('Precipitation (mm)', color=color)  # we already handled the x-label with ax1
ax2.plot(df_plot.index, df_plot["prcp (mm/day)"], color=color)
ax2.tick_params(axis='y', labelcolor=color)
ax2.set_ylim(100, 0)

axmax = ax1.get_ylim()[1].copy()
# parallelogram for each event -- base is the recharge event, top is the precip event
for _, row in event_table.iterrows():
    if pd.notna(row["recharge_start_date"]) and pd.notna(row["recharge_end_date"]):
        # coordinates of the parallelogram
        x = [row["recharge_end_date"], row["recharge_start_date"], row["precip_start_date"], row["precip_end_date"]]
        y = [0, 0, axmax, axmax]
        ax1.fill(x, y, color='gray', alpha=0.2)

plt.show()

# %%


[(799, Timestamp('2021-02-27 00:00:00'), Timestamp('2021-02-27 00:00:00'), 1)]


In [17]:
eligible_wells = df_ne_wells[df_ne_wells["record_length"] >= 5].copy()
print(f"Wells with ≥5 year records: {len(eligible_wells)}")

# Pick 10 at random, with a fixed seed so this is reproducible if you re-run it
selected_wells = eligible_wells.sample(n=10, random_state=42)
print(selected_wells[["usgs_id", "record_length", "start_date", "end_date"]])

NameError: name 'df_ne_wells' is not defined